# 01 — Test the base Qwen3-8B (no fine-tuning)

The first of three notebooks — run them in order, each in its own Colab session:

| | Notebook | What it does |
|---|---|---|
| **01** | **this one** | plain Qwen3-8B answers the validation examples → `eval_base.jsonl` |
| 02 | `02_train_qlora.ipynb` | trains the QLoRA adapter → `adapters/qwen3-8b-<version>/` |
| 03 | `03_eval_finetuned_model.ipynb` | Qwen3-8B + the adapter answers the same examples, and compares with this run |

This gives the **"Qwen3 8B, not fine-tuned"** row of the thesis comparison. Nothing is trained here.

**Before you start:** *Runtime → Change runtime type → **T4 GPU***, then run the steps in order. If Colab disconnects part-way, just run everything again: the answers already saved on Drive are kept and the test continues after the last one.

## Step 1 — Check the GPU
Confirms a GPU is attached and which one. On a T4 you should see **Tesla T4, compute capability 7.5**. The T4 has no real bfloat16 support, so float16 is used on it (bfloat16 only on newer GPUs such as A100/L4).

In [1]:
!nvidia-smi -L
import torch
print("CUDA available:", torch.cuda.is_available())
major, minor = torch.cuda.get_device_capability()
print(f"GPU: {torch.cuda.get_device_name(0)}  |  compute capability {major}.{minor}")
print("Precision used:", "bfloat16" if major >= 8 else "float16 (this GPU has no real bfloat16)")

GPU 0: Tesla T4 (UUID: GPU-69c7cf9f-a85b-12a1-3d93-4c14ea724034)
CUDA available: True
GPU: Tesla T4  |  compute capability 7.5
Precision used: float16 (this GPU has no real bfloat16)


## Step 2 — Install the libraries
- **transformers** — loads Qwen3 (pinned to 4.x, 4.56 or newer — this code uses the 4.x API).
- **peft** — the LoRA adapter. **bitsandbytes** — 4-bit loading (the "Q" in QLoRA).
- **accelerate**, **datasets** — GPU placement and the data container.

Takes 1–2 minutes. If Colab asks you to **restart the session**, do it, then continue with Step 3.

In [2]:
!pip install -q "transformers>=4.56,<5.0" "peft>=0.13" "bitsandbytes>=0.43" "accelerate>=0.34" datasets
import transformers, peft, bitsandbytes
print("transformers", transformers.__version__, "| peft", peft.__version__, "| bitsandbytes", bitsandbytes.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 73.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 4.57.6 | peft 0.21.1 | bitsandbytes 0.50.2


## Step 3 — Get the data and choose where results go
Downloads the repository (the data is in its `data/` folder) and sets the folders. **Keep these settings the same in all three notebooks** — notebook 03 finds the adapter from 02 and the base results from 01 through them.
- **`DATA_VERSION`** — which data: `"v2"` uses `questions_v2_train.jsonl` / `questions_v2_val.jsonl`.
- **`USE_DRIVE = True`** *(recommended)* — outputs go to `MyDrive/kqml_finetune/` and survive a disconnect. With `False` they stay on Colab's disk and are lost when the session ends.

Every output of a version goes in its own folder, `qwen3-8b-<version>`, so v1 and v2 never mix.

In [3]:
from pathlib import Path

DATA_VERSION = "v2"                  # questions_<version>_train.jsonl / _val.jsonl
USE_DRIVE    = True                  # False = Colab disk only (lost when the session ends)
REPO         = "https://github.com/ishancoderr/kqml-geo-qwen3-qlora.git"
BRANCH       = "main"

!rm -rf /content/kqml-geo-qwen3-qlora
!git clone -q --depth 1 --branch {BRANCH} {REPO} /content/kqml-geo-qwen3-qlora
DATA_DIR   = Path("/content/kqml-geo-qwen3-qlora/data")
TRAIN_FILE = DATA_DIR / f"questions_{DATA_VERSION}_train.jsonl"
VAL_FILE   = DATA_DIR / f"questions_{DATA_VERSION}_val.jsonl"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/kqml_finetune")
else:
    WORK_DIR = Path("/content/kqml_finetune")

RUN_NAME       = f"qwen3-8b-{DATA_VERSION}"
ADAPTER_DIR    = WORK_DIR / "adapters" / RUN_NAME       # written by 02, read by 03
CHECKPOINT_DIR = WORK_DIR / "checkpoints" / RUN_NAME    # 02 only: one checkpoint per epoch
RESULTS_DIR    = WORK_DIR / "results" / RUN_NAME        # eval_base.jsonl, eval_tuned.jsonl, training log
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for f in (TRAIN_FILE, VAL_FILE):
    print(("found    " if f.exists() else "MISSING  ") + str(f))
print("outputs go to:", WORK_DIR, "| run:", RUN_NAME)

Mounted at /content/drive
found    /content/kqml-geo-qwen3-qlora/data/questions_v2_train.jsonl
found    /content/kqml-geo-qwen3-qlora/data/questions_v2_val.jsonl
outputs go to: /content/drive/MyDrive/kqml_finetune | run: qwen3-8b-v2


## Step 4 — Settings
- **MAX_NEW_TOKENS** — the longest answer allowed when testing, per task.
- **LIMIT** — `None` answers every validation example. Set it to e.g. `10` for a quick trial run first; later runs continue from where the file stops.

In [4]:
import torch

MODEL_NAME     = "Qwen/Qwen3-8B"
MAX_NEW_TOKENS = {"classify": 32, "extract": 384, "sql": 512}
LIMIT          = None            # e.g. 10 for a quick trial; None = all validation examples

DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print("compute dtype:", DTYPE)

compute dtype: torch.float16


## Step 5 — Read the data
Each line of the JSONL is **one LLM call** of the agent: system prompt, user message, the correct answer, and a `meta` block. From `meta` we keep the **task** (classify / extract / sql), the question id and the category, so the scores can be shown per task and per category.

In [5]:
import json
from collections import Counter

def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            msgs = rec["messages"]
            assert [m["role"] for m in msgs] == ["system", "user", "assistant"], msgs
            meta = rec.get("meta", {})
            rows.append({"task": meta.get("task", "all"), "qid": meta.get("qid", ""),
                         "category": meta.get("category", ""),
                         "prompt_messages": msgs[:2], "answer": msgs[2]["content"]})
    return rows

val_rows = load_jsonl(VAL_FILE)
print("val:", len(val_rows), dict(Counter(r["task"] for r in val_rows)))

val: 101 {'classify': 31, 'extract': 29, 'sql': 41}


## Step 6 — Tokenizer
The prompt is rendered with **Qwen3's own chat template, thinking turned off** (`enable_thinking=False`) — exactly the prompt the model gets when the agent uses it.

In [6]:
from transformers import AutoTokenizer

END = "<|im_end|>"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
END_ID = tokenizer.convert_tokens_to_ids(END)

def prompt_text(row):
    return tokenizer.apply_chat_template(row["prompt_messages"], tokenize=False,
                                         add_generation_prompt=True, enable_thinking=False)

print("--- end of one prompt ---\n..." + prompt_text(val_rows[0])[-200:])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

--- end of one prompt ---
...ATED"}

"What is the population of Paris?"
→ {"query_type":"UNRELATED"}
<|im_end|>
<|im_start|>user
How many residents did Brandenburg have in 2020?<|im_end|>
<|im_start|>assistant
<think>

</think>




## Step 7 — Load Qwen3-8B in 4-bit
Downloads the model (about **16 GB**, a few minutes, once per session) and loads it **quantised to 4-bit NF4**, about 6 GB of GPU memory. Generation is switched to plain greedy decoding, so every answer is deterministic.

If the session **crashes while loading**, it ran out of *system RAM* (free Colab has about 12 GB), not GPU memory — restart the session and run again from Step 1.

In [7]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=DTYPE,
    ),
    device_map={"": 0},
    dtype=DTYPE,
)
gen = model.generation_config
gen.do_sample, gen.temperature, gen.top_p, gen.top_k = False, None, None, None
print(f"loaded — GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

...

model-00003-of-00005.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/1.24G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

loaded — GPU memory in use: 6.1 GB


## Step 8 — Testing helpers
How each validation example is tested (nothing runs yet):
- The model answers with greedy decoding.
- **classify / extract:** the answer must be valid JSON and **exactly equal** to the correct JSON.
- **sql:** the statement is compared after ignoring spacing and letter case — strict: a correct statement written differently counts as wrong (running it on the database is the real check, later).
- **valid JSON** is reported separately — whether the model at least kept the output format.

**Saved as it goes:** every answer is appended to `eval_<tag>.jsonl` on Drive the moment it is made. If Colab disconnects, run the notebook again — it continues after the last saved answer instead of starting over. To start a run over, delete that file (or set `restart=True`).

In [8]:
import re, time, statistics

def clean(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
    fenced = re.match(r"^```[a-zA-Z]*\s*(.*?)\s*```$", text, flags=re.S)
    return fenced.group(1).strip() if fenced else text

def parse(text):
    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        return None

def norm_sql(sql):
    return re.sub(r"\s+", " ", sql.strip().rstrip(";")).lower()

def score(task, gold, pred):
    p, g = parse(pred), parse(gold)
    if p is None:
        return False, False                      # not even valid JSON
    if task == "sql":
        ok = isinstance(p, dict) and isinstance(p.get("sql"), str) and norm_sql(p["sql"]) == norm_sql(g["sql"])
        return True, ok
    return True, p == g

@torch.no_grad()
def answer(row):
    inputs = tokenizer(prompt_text(row), return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS.get(row["task"], 512), do_sample=False,
                         eos_token_id=END_ID, pad_token_id=tokenizer.pad_token_id)
    return clean(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

def print_table(tag, results, key="task", keys=("classify", "extract", "sql")):
    print(f"\n{tag} — by {key}\n  {key:<28}{'n':>4}{'valid JSON':>12}{'correct':>10}{'s/answer':>10}")
    for k in [*keys, "TOTAL"]:
        g = results if k == "TOTAL" else [r for r in results if r[key] == k]
        if g:
            print(f"  {k:<28}{len(g):>4}{sum(r['valid_json'] for r in g) / len(g):>12.0%}"
                  f"{sum(r['correct'] for r in g) / len(g):>10.0%}{statistics.mean(r['seconds'] for r in g):>10.1f}")

def load_results(tag):
    path = RESULTS_DIR / f"eval_{tag}.jsonl"
    if not path.exists():
        return []
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]

def run_eval(tag, restart=False):
    model.eval()
    model.config.use_cache = True
    rows = val_rows[:LIMIT] if LIMIT else val_rows
    path = RESULTS_DIR / f"eval_{tag}.jsonl"
    if restart and path.exists():
        path.unlink()
    done = load_results(tag)
    # the saved answers must belong to these same examples, in this order
    if [r["n"] for r in done] != list(range(1, len(done) + 1)) or \
       any(r["qid"] != val_rows[r["n"] - 1]["qid"] for r in done):
        raise RuntimeError(f"{path.name} does not match this validation file — delete it or use restart=True")
    if done:
        print(f"continuing: {len(done)} of {len(rows)} already answered")
    with open(path, "a", encoding="utf-8") as f:
        for i, row in enumerate(rows[len(done):], len(done) + 1):
            t0 = time.perf_counter()
            pred = answer(row)
            secs = time.perf_counter() - t0
            valid, correct = score(row["task"], row["answer"], pred)
            rec = {"n": i, "qid": row["qid"], "task": row["task"], "category": row["category"],
                   "gold": row["answer"], "pred": pred, "valid_json": valid, "correct": correct,
                   "seconds": round(secs, 2)}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            f.flush()                                       # on Drive at once — survives a disconnect
            done.append(rec)
            print(f"[{i}/{len(rows)}] {row['qid']:<4} {row['task']:<8} {'ok ' if correct else 'BAD'} {secs:5.1f}s")
    print_table(tag, done)
    print_table(tag, done, key="category", keys=sorted({r["category"] for r in done}))
    print("saved:", path)
    return done

print("helpers ready")

helpers ready


## Step 9 — Test the base model
Plain Qwen3-8B on the validation examples, scored per task and per category. Roughly 5–20 seconds per example (the base model often writes longer answers than needed). Re-running this step continues where it stopped.

In [9]:
base_results = run_eval("base")

[1/101] d10  classify ok    4.2s
[2/101] d10  extract  ok    4.3s
[3/101] d10  sql      ok    7.7s
[4/101] d20  classify ok    2.7s
[5/101] d20  extract  ok    6.8s
[6/101] d20  sql      ok    8.2s
[7/101] d20  sql      ok    8.3s
[8/101] d20  sql      ok    8.5s
[9/101] d30  classify ok    3.0s
[10/101] d30  extract  ok    4.8s
[11/101] d30  sql      ok    8.0s
[12/101] d40  classify ok    3.2s
[13/101] d40  extract  ok    6.0s
[14/101] d40  sql      ok    9.7s
[15/101] d50  classify ok    3.2s
[16/101] d50  extract  ok    4.8s
[17/101] d50  sql      ok    8.9s
[18/101] d60  classify ok    3.1s
[19/101] d60  extract  ok    4.6s
[20/101] d60  sql      ok    8.5s
[21/101] g10  classify ok    3.1s
[22/101] g10  extract  ok    2.4s
[23/101] g20  classify ok    3.1s
[24/101] g20  extract  ok    2.5s
[25/101] g30  classify ok    3.1s
[26/101] g30  extract  ok    2.6s
[27/101] g40  classify ok    3.2s
[28/101] g40  extract  ok    4.8s
[29/101] g40  sql      ok    5.4s
[30/101] g50  classify 

## Step 10 — Look at the mistakes
What the base model gets wrong, with the correct answer underneath — typical are extra text around the JSON, other field names, or SQL with literal values instead of `:placeholders`.

In [10]:
def show_mistakes(results, limit=40):
    wrong = [r for r in results if not r["correct"]]
    print(f"{len(wrong)} of {len(results)} wrong\n")
    for r in wrong[:limit]:
        print(f"{r['qid']} · {r['task']} · {r['category']}")
        print("  model:  ", r["pred"][:400])
        print("  correct:", r["gold"][:400])
        print()

show_mistakes(base_results)

17 of 101 wrong

o60 · extract · SPATIAL_OPERATION
  model:   {
  "operation": "Intersection",
  "spatial": ["Hessen", "Bayern"],
  "entity_type": "state",
  "target_entity": "city"
}
  correct: {"operation": "Intersection", "spatial": ["Hessen", "Bayern"], "entity_type": "state", "distance_km": null, "target_entity": "city"}

o70 · sql · SPATIAL_OPERATION
  model:   {"sql": "WITH shapes AS (SELECT ST_GeomFromText(ST_GeomFromText(:wkt_a, :srid_a)) AS geom_a, ST_GeomFromText(:wkt_b, :srid_b) AS geom_b), combined AS (SELECT ST_Union(geom_a, geom_b) AS combined_geom FROM shapes) SELECT ST_AsText(combined_geom) AS wkt, SUM(state_demographics.live_births) AS total_live_births FROM combined JOIN states ON states.geo_shape = combined.combined_geom JOIN state_demograp
  correct: {"sql": "SELECT ST_AsText(ST_Union(ST_GeomFromText(:wkt_a, :srid_a), ST_GeomFromText(:wkt_b, :srid_b))) AS wkt"}

o70 · sql · DIRECT_LOOKUP
  model:   {"sql": "SELECT s.state_name AS entity_name, sd.stat_year AS year, 

## Step 11 — Download the results (only needed when `USE_DRIVE = False`)
With Drive the file is already in `MyDrive/kqml_finetune/results/qwen3-8b-<version>/`.

In [11]:
import shutil
from google.colab import files

bundle = Path("/content/bundle")
shutil.rmtree(bundle, ignore_errors=True)
shutil.copytree(RESULTS_DIR, bundle / "results")
zip_path = shutil.make_archive(f"/content/{RUN_NAME}-base", "zip", bundle)
print(f"{Path(zip_path).name}: {Path(zip_path).stat().st_size / 1e6:.0f} MB")
files.download(zip_path)

qwen3-8b-v2-base.zip: 0 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>